# 02 Preprocessing

In [33]:
# One section per preprocessing decision: question, data, decision, reason. The order
# follows the course (Project Discussions, slide 10): the embedding comes first,
# because tokenization, lowercasing, stemming, vocabulary and truncation have to fit it.
import sys
sys.path.insert(0, "..")  # makes src importable from notebooks/

import nltk
import pandas as pd
import re

from collections import Counter
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import NLTKWordTokenizer
from src.data import TEXT_COLUMNS, load_splits
from src.preprocessing import (
    DATA_DIR, END_CHARS, SEP_IDX, UNK_IDX, encode, load_glove, tokenize
)

pd.set_option("display.max_colwidth", None)
splits = load_splits()
train = splits["train"]
# typographic apostrophe replaced as in tokenize() (01_data_exploration.ipynb)
train_texts = [t.replace("’", "'") for col in TEXT_COLUMNS for t in train[col]]

## 1. Embedding

In [34]:
# Which pretrained embedding? The task allows word2vec, GloVe or fastText
# (Project 1 description). The lecture says GloVe "generally performs as well as
# word2vec, choice is personal preference (or empirical question)" (Word Embeddings,
# slide 21).
#
# GloVe 6B (Wikipedia 2014 + Gigaword 5), 300 dimensions, no comparison with
# word2vec or fastText.
# As good as word2vec according to the lecture. It was trained on lowercased
# text and knows 400k words, which keeps tokenization simple and leaves very few of our
# words without a vector (section 2). 300 is the largest size (50/100/200/300), the
# same as word2vec and fastText. Of the GloVe versions (github.com/stanfordnlp/glove),
# 6B is the classic one and on the official mirror (huggingface.co/stanfordnlp/glove).
# The 2024 version is the same kind of text at twice the size, Common Crawl 840B is
# cased and Twitter is a different kind of text.
#
# The GloVe file has one line per word: the word, then its 300 numbers. Here we only
# need the words.
glove_words = set()
with open(DATA_DIR / "glove.6B.300d.txt", encoding="utf-8") as f:
    for line in f:
        glove_words.add(line.split(" ", 1)[0])

uppercase = sum(any(c.isupper() for c in w) for w in glove_words)
print(f"GloVe words: {len(glove_words):,}")
print("words with an uppercase letter:", uppercase)

GloVe words: 400,001
words with an uppercase letter: 0


## 2. Tokenization

In [35]:
# How do we split a sentence into tokens? GloVe only has vectors for whole
# words, so subword tokenization (BPE, NLP Pipeline slides 10-12) does not fit.
# Word-level candidates (NLP Pipeline, slides 8-13):
#   whitespace  text.split(), the simplest one
#   nltk        nltk.word_tokenize from the lecture (slide 13): splits sentences first,
#               then words with Penn Treebank rules, like GloVe's training text
#               (didn't -> did n't)
# Tokens should look like the words GloVe was trained on, otherwise they get no vector.
# Both lowercase, because GloVe has no uppercase words (section 1).
tokenizers = {
    "whitespace": lambda text: text.lower().split(),
    "nltk": lambda text: [t.lower() for t in nltk.word_tokenize(text)],
}

example = next(t for t in train_texts if "n't" in t and "," in t)
print(example)
for name, tok in tokenizers.items():
    print(f"{name:>10}: {tok(example)}")

Cody had a black cat, but she didn't like the cat.
whitespace: ['cody', 'had', 'a', 'black', 'cat,', 'but', 'she', "didn't", 'like', 'the', 'cat.']
      nltk: ['cody', 'had', 'a', 'black', 'cat', ',', 'but', 'she', 'did', "n't", 'like', 'the', 'cat', '.']


In [36]:
# How many training tokens have no GloVe vector with each tokenizer?
def unknown_row(tokens: list[str], top: int) -> dict:
    unknown = Counter(t for t in tokens if t not in glove_words)
    return {
        "tokens": len(tokens),
        "unknown tokens": sum(unknown.values()),
        "unknown %": round(100 * sum(unknown.values()) / len(tokens), 2),
        "most frequent unknown": [w for w, _ in unknown.most_common(top)],
    }


rows = {
    name: unknown_row([t for text in train_texts for t in tok(text)], 6)
    for name, tok in tokenizers.items()
}
pd.DataFrame.from_dict(rows, orient="index")

,tokens,unknown tokens,unknown %,most frequent unknown
whitespace,573334,81660,14.24,"[didn't, it., day., couldn't, him., home.]"
nltk,655445,1020,0.16,"[mturk, .., selfie, zaiden, a+, antonela]"


In [37]:
# "n't" has its own GloVe vector, ".." does not. ".." comes from typos with a double
# full stop:
print("n't" in glove_words, ".." in glove_words, "..." in glove_words)
print([t for t in train_texts if re.search(r"(?<!\.)\.\.(?!\.)", t)][:4])

True False True
['and they started to enjoy the animals in the safari..', 'My friend send me a bouquet ,but the delivery guy delivered it to wrong person..', 'they planned to go to a steak restaurant..', "Keith's parents took the family car for the trip.."]


In [38]:
# Why word_tokenize and not NLTK's word tokenizer alone: about 1% of the hypotheses
# have two sentences (01_data_exploration.ipynb). Without splitting the sentences
# first, the inner full stop stays glued to the word.
two_sentences = (
    "My neighbor asked me to take care of their dog while they went on vacation. "
    "I agreed."
)
print(NLTKWordTokenizer().tokenize(two_sentences))
print(nltk.word_tokenize(two_sentences))

['My', 'neighbor', 'asked', 'me', 'to', 'take', 'care', 'of', 'their', 'dog', 'while', 'they', 'went', 'on', 'vacation.', 'I', 'agreed', '.']
['My', 'neighbor', 'asked', 'me', 'to', 'take', 'care', 'of', 'their', 'dog', 'while', 'they', 'went', 'on', 'vacation', '.', 'I', 'agreed', '.']


In [39]:
# We use nltk.word_tokenize, after replacing the typographic apostrophe.
# The standard rule-based tokenizer from the lecture. It follows the same Penn
# Treebank conventions as GloVe's training text, so only 0.16% of the training tokens
# have no vector (whitespace: 14%), and no rules have to be written by hand.
#
# tokenize() in src/preprocessing.py, with the decisions of sections 3 and 6:
print(tokenize("Iris catch down Lisa’s phone. She didn't care!"))

['iris', 'catch', 'down', 'lisa', "'s", 'phone', '.', 'she', 'did', "n't", 'care', '!']


## 3. Lowercasing

In [40]:
# Lowercase everything? Depends on the embedding (Project Discussions,
# slide 10: "Lowercase -> model"). GloVe 6B has no uppercase words (section 1), so
# every token with a capital letter would get no vector.
cased = [t for text in train_texts for t in nltk.word_tokenize(text)]
rows = {
    "as written": unknown_row(cased, 8),
    "lowercased": unknown_row([t.lower() for t in cased], 8),
}
pd.DataFrame.from_dict(rows, orient="index")

,tokens,unknown tokens,unknown %,most frequent unknown
as written,655445,82407,12.57,"[I, The, He, She, My, It, A, They]"
lowercased,655445,1020,0.16,"[mturk, .., selfie, zaiden, a+, antonela, fariah, myah]"


In [41]:
# Lowercase all tokens.
# GloVe 6B was trained on lowercased text. Without lowercasing, 12.6% of the
# training tokens would have no vector: every sentence start ("The", "He") and every
# name. What only lies in the case ("US" vs. "us") is lost, but GloVe could not
# represent it anyway.

## 4. Stemming / lemmatization

In [42]:
# Reduce words to a common form before the lookup? (NLP Pipeline, slides
# 15-17)
#   stemming       cuts off word endings by rules: houses -> hous (PorterStemmer)
#   lemmatization  looks up the base form in a dictionary: houses -> house (WordNet)
# Project Discussions slide 10 marks both as "depends on the model".
nltk.download("wordnet", quiet=True)
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = [
    "houses", "flies", "studied", "lost", "losing",
    "went", "was", "better", "happily", "cookies",
]
forms = pd.DataFrame(
    {
        "word": words,
        "stem": [stemmer.stem(w) for w in words],
        "lemma": [lemmatizer.lemmatize(w) for w in words],
    }
).set_index("word")

# The GloVe file is sorted by frequency (line 1 = most frequent word). Being in GloVe
# is not enough: "hous" exists, but only as a rare token with a poorly learned vector.
needed = set(words) | set(forms["stem"]) | set(forms["lemma"])
rank = {}
with open(DATA_DIR / "glove.6B.300d.txt", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        w = line.split(" ", 1)[0]
        if w in needed:
            rank[w] = line_no
forms["rank word"] = [rank.get(w) for w in forms.index]
forms["rank stem"] = [rank.get(w) for w in forms["stem"]]
forms["rank lemma"] = [rank.get(w) for w in forms["lemma"]]
forms

,stem,lemma,rank word,rank stem,rank lemma
word,,,,,
houses,hous,house,1632,261225.0,167.0
flies,fli,fly,8483,186249.0,2362.0
studied,studi,studied,2600,90635.0,2600.0
lost,lost,lost,403,403.0,403.0
losing,lose,losing,1530,1896.0,1530.0
went,went,went,389,389.0,389.0
was,wa,wa,16,15278.0,15278.0
better,better,better,440,440.0,440.0
happily,happili,happily,12442,NaN,12442.0


In [43]:
# How many training tokens have no GloVe vector after each variant?
tokens = [t for text in train_texts for t in tokenize(text)]
variants = {
    "none": lambda t: t,
    "stemming": stemmer.stem,
    "lemmatization": lemmatizer.lemmatize,
}
rows = {name: unknown_row([f(t) for t in tokens], 8) for name, f in variants.items()}
pd.DataFrame.from_dict(rows, orient="index")[["unknown %", "most frequent unknown"]]

,unknown %,most frequent unknown
none,0.17,"[mturk, selfie, zaiden, a+, antonela, fariah, myah, shalil]"
stemming,7.31,"[decid, famili, becaus, realli, realiz, movi, everi, excit]"
lemmatization,0.20,"[cooky, mturk, selfie, zaiden, a+, antonela, fariah, myah]"


In [44]:
# Does it remove differences between the two hypotheses? Count the pairs where words
# that differ between h1 and h2 ("lost" vs. "losing") get the same form.
for name in ["stemming", "lemmatization"]:
    f = variants[name]
    merged = []
    for h1, h2 in zip(train["hypothesis_1"], train["hypothesis_2"]):
        t1, t2 = tokenize(h1), tokenize(h2)
        before = set(t1) ^ set(t2)
        after = {f(t) for t in t1} ^ {f(t) for t in t2}
        if len(after) < len(before):
            merged.append((h1, h2))
    share = 100 * len(merged) / len(train)
    print(f"{name}: {len(merged)} pairs ({share:.1f}%) lose a difference")
    for h1, h2 in merged[:3]:
        print("   ", h1, "|", h2)

stemming: 1292 pairs (7.7%) lose a difference
    The man didn't knock him out. | The man knocked him out with a bat.
    Amanda went to the dealership to look at cars. | Amanda could not afford a car.
    He screamed out loud inside. | He couldn't even scream out loud.
lemmatization: 413 pairs (2.5%) lose a difference
    Amanda went to the dealership to look at cars. | Amanda could not afford a car.
    Bam's date got upset because she didn't like action movies. | Bam took her to an action movie.
    The golden retriever wore it as a necklace. | Bay's ring was on her desk the next morning.


In [45]:
# No stemming, no lemmatization.
# GloVe has its own vector for every word form ("houses", "studied", "lost"),
# and similar forms already have similar vectors. Stemming makes 7.3% of the tokens
# unknown (made-up forms like "hous", "famili"), and both remove differences between
# the two hypotheses that the model should see (knock/knocked, car/cars).

## 5. Stopwords

In [46]:
# Keep stopwords.
# Discussed in class (Project Discussions, slide 10: stopword removal "no").
# Small words are often exactly what makes a hypothesis wrong: in 2.2% of the training
# pairs the two hypotheses differ only in stopwords ("Amanda is not easily amused." /
# "Amanda is easily amused."), so removing them would make h1 and h2 identical
# (01_data_exploration.ipynb, stopwords).

## 6. Punctuation

In [47]:
# Remove punctuation tokens? Project Discussions slide 10 recommends it
# (depends on the dataset). Here a token counts as punctuation if it has no letter or
# digit ("n't" and "'s" are words).
def is_punct(token: str) -> bool:
    return not any(c.isalnum() for c in token)


nltk_tokenize = tokenizers["nltk"]
all_tokens = [t for text in train_texts for t in nltk_tokenize(text)]
punct = Counter(t for t in all_tokens if is_punct(t))
n_punct = sum(punct.values())
share = 100 * n_punct / len(all_tokens)
print(f"punctuation tokens: {n_punct:,} of {len(all_tokens):,} ({share:.1f}%)")
table = [(t, n, t in glove_words) for t, n in punct.most_common()]
pd.DataFrame(table, columns=["token", "count", "in GloVe"]).set_index("token").T

punctuation tokens: 73,814 of 655,445 (11.3%)


token,.,",",!,$,``,','',-,?,&,..,;,%,--,{,},:,...,`,\
count,65516,6061,1777,116,102,82,37,29,24,16,14,11,8,5,4,4,3,3,1,1
in GloVe,True,True,True,True,True,True,True,True,True,True,False,True,True,True,True,True,True,True,True,True


In [48]:
# Architecture 1 averages the word vectors of a sentence. Nearly every sentence ends
# with ".", so the same vector goes into almost every average; in a sentence of 9
# words it is 10% of the result.
sentence = train["observation_1"][0]
print(nltk_tokenize(sentence))
print(f'"." makes up {100 / len(nltk_tokenize(sentence)):.0f}% of this average')

['a', 'baby', 'panda', 'was', 'lost', 'in', 'the', 'woods', '.']
"." makes up 11% of this average


In [49]:
# Does punctuation ever make the difference between the two hypotheses? Pairs that are
# the same without punctuation but differ with it:
def words_only(text: str) -> list[str]:
    return [t for t in nltk_tokenize(text) if not is_punct(t)]


pairs = zip(train["hypothesis_1"], train["hypothesis_2"], train["label"])
only_punct = [
    (h1, h2, label)
    for h1, h2, label in pairs
    if nltk_tokenize(h1) != nltk_tokenize(h2) and words_only(h1) == words_only(h2)
]
print(len(only_punct), "pairs differ only in punctuation")
for example in only_punct[:5]:
    print("   ", example)

0 pairs differ only in punctuation


In [50]:
# The variant chosen after looking at the sentence endings (01_data_exploration.ipynb):
# strip only ". , / \" from the end of each text and keep all other punctuation. What
# tokenize() does with some real endings:
print("stripped from the end:", list(END_CHARS))
endings = [
    "Jenny woke up excited for her big picnic today!",
    "Heather wanted to challenge her friends to a mud run,",
    "To this day I like candlepin bowling/",
    'Irene was sure it would be easy.".',
    "He left immediately.,.",
    two_sentences,
]
for text in endings:
    print(tokenize(text))

stripped from the end: ['.', ',', '/', '\\']
['jenny', 'woke', 'up', 'excited', 'for', 'her', 'big', 'picnic', 'today', '!']
['heather', 'wanted', 'to', 'challenge', 'her', 'friends', 'to', 'a', 'mud', 'run']
['to', 'this', 'day', 'i', 'like', 'candlepin', 'bowling']
['irene', 'was', 'sure', 'it', 'would', 'be', 'easy', '.', "''"]
['he', 'left', 'immediately']
['my', 'neighbor', 'asked', 'me', 'to', 'take', 'care', 'of', 'their', 'dog', 'while', 'they', 'went', 'on', 'vacation', '.', 'i', 'agreed']


In [51]:
# Which punctuation tokens are left, and how many tokens have no GloVe vector now?
tokens = [t for text in train_texts for t in tokenize(text)]
remaining = Counter(t for t in tokens if is_punct(t))
n_left = sum(remaining.values())
share = 100 * n_left / len(tokens)
print(f"punctuation tokens left: {n_left:,} of {len(tokens):,} ({share:.1f}%)")
print(remaining.most_common())
row = unknown_row(tokens, 8)
print(f"unknown tokens: {row['unknown %']:.2f}%", row["most frequent unknown"])

punctuation tokens left: 8,546 of 590,177 (1.4%)
[(',', 6020), ('!', 1777), ('.', 305), ('$', 116), ("''", 91), ("'", 82), ('``', 48), ('-', 29), ('?', 24), ('&', 16), (';', 11), ('%', 8), ('--', 5), ('{', 4), ('}', 4), (':', 3), ('...', 2), ('`', 1)]
unknown tokens: 0.17% ['mturk', 'selfie', 'zaiden', 'a+', 'antonela', 'fariah', 'myah', 'shalil']


In [52]:
# Strip ". , / \" from the end of every text, keep all other punctuation.
# Every hypothesis ends with "." (01_data_exploration.ipynb, sentence endings),
# so the final full stop tells nothing but would go into almost every average. A final
# "," "/" or "\" is a cut-off or broken ending (12 texts). "?" and quotes are rare and
# part of the content, "!" at the end of observation_2 (10%) shows the tone of the
# ending, and commas inside a sentence and "$" ("$5") carry meaning. No hypothesis pair
# differs only in punctuation, so this does not touch what tells h1 and h2 apart.

## 7. Unknown and rare words

In [53]:
# What happens with words that have no GloVe vector? (Project Discussions,
# slide 10: "Handling unknown/rare words (map to UNK)")
# From 01_data_exploration.ipynb: 843 different unknown tokens, 1,137 occurrences
# (0.17%), mostly rare first names, typos and hyphenated words. Splitting the unknown
# hyphenated words leaves 588 different unknown words, 851 occurrences (0.125%).
#
# encode() in src/preprocessing.py on real sentences: the name becomes <unk>, the
# hyphenated word is split.
vocab, _ = load_glove()
id_to_word = {i: w for w, i in vocab.items()}
for word in ["Zaiden", "bb-gun"]:
    text = next(t for t in train_texts if word in t)
    print(text)
    print("   tokens: ", tokenize(text))
    print("   encoded:", [id_to_word[i] for i in encode(text, vocab)])

There was a boy name Zaiden.
   tokens:  ['there', 'was', 'a', 'boy', 'name', 'zaiden']
   encoded: ['there', 'was', 'a', 'boy', 'name', '<unk>']
His friend wanted to play with his new bb-gun.
   tokens:  ['his', 'friend', 'wanted', 'to', 'play', 'with', 'his', 'new', 'bb-gun']
   encoded: ['his', 'friend', 'wanted', 'to', 'play', 'with', 'his', 'new', 'bb', 'gun']


In [54]:
# How many tokens end up as <unk>, over all splits?
ids = [
    i
    for split in splits.values()
    for col in TEXT_COLUMNS
    for text in split[col]
    for i in encode(text, vocab)
]
n_unk = sum(i == UNK_IDX for i in ids)
print(f"{n_unk:,} of {len(ids):,} ids are <unk> ({100 * n_unk / len(ids):.3f}%)")

851 of 680,613 ids are <unk> (0.125%)


In [55]:
# An unknown word stays in the text as <unk>; unknown hyphenated words are
# split into their parts first. Rare words that have a GloVe vector are kept.
# The word keeps its position, but its content is ignored: <unk> has a zero
# vector, architecture 1 leaves it out of the average, and the RNN still sees that
# there was a word ("<unk> went home" keeps the subject). This is what the course
# suggests ("map to UNK", Project Discussions slide 10), and it concerns only 0.125%
# of the tokens. Splitting keeps the meaning of 255 of the 843 unknown words ("bb-gun"
# -> "bb" "gun"). Rare words keep their pretrained vector, which does not get worse by
# being rare in our data.

## 8. Broken entries

In [56]:
# Are there broken entries that should be removed? The empty hypotheses
# ("{}." and ".") from 01_data_exploration.ipynb. Which rule catches exactly those?
hypotheses = [
    (name, i, col, row[col], row["label"])
    for name, split in splits.items()
    for i, row in enumerate(split)
    for col in ["hypothesis_1", "hypothesis_2"]
]
rules = {
    "fewer than 3 characters": lambda h: len(h.strip()) < 3,
    "at most 2 tokens": lambda h: len(tokenize(h)) <= 2,
    "no letter at all": lambda h: not any(c.isalpha() for c in h),
}
for name, rule in rules.items():
    print(f"{name}:", [h for _, _, _, h, _ in hypotheses if rule(h)])

fewer than 3 characters: ['.', '.']
at most 2 tokens: ['shesaid.', '{}.', 'NONSENSICAL.', 'No sense.', '{}.', '.', '{}.', '.', '{}.', 'not probable.', 'Aura overslept.', 'Ideas nextbdoor.', 'Mary died.']
no letter at all: ['{}.', '{}.', '.', '{}.', '.', '{}.']


In [57]:
# The empty hypotheses with their label (which hypothesis is right): the empty one is
# always the wrong one
for name, i, col, h, label in hypotheses:
    if not any(c.isalpha() for c in h):
        print(f"{name} #{i}: {col} = {h!r}, label = {label}")

train #2260: hypothesis_2 = '{}.', label = 1
train #6855: hypothesis_1 = '{}.', label = 2
train #7734: hypothesis_2 = '.', label = 1
train #8765: hypothesis_1 = '{}.', label = 2
train #8768: hypothesis_1 = '.', label = 2
train #11355: hypothesis_2 = '{}.', label = 1


In [58]:
# Other broken things from 01_data_exploration.ipynb: broken encoding in 3
# observations, line breaks in 6 texts (the tokenizer treats them like spaces)
texts = [t for split in splits.values() for col in TEXT_COLUMNS for t in split[col]]
print([t for t in texts if "�" in t])
print(tokenize("It was packed and the sand was\ncrowded and everyone was unfriendly."))

["Jo��o was frustrated at his country's government.", 'He opened the envelope and was disappointed he hadn���t been accepted.', 'Alex took his clothes to his Mom���s house for her to clean.']
['it', 'was', 'packed', 'and', 'the', 'sand', 'was', 'crowded', 'and', 'everyone', 'was', 'unfriendly']


In [59]:
# Remove the 6 training examples with an empty hypothesis (rule: no letter
# at all, "{}." or "."), leave everything else as it is.
# In all 6 the empty hypothesis is the wrong one, so they would only teach
# "an empty hypothesis is wrong", which never happens in validation or test. "No letter
# at all" catches exactly these 6; "fewer than 3 characters" misses "{}.", "at most 2
# tokens" would also remove real hypotheses like "Mary died.". The broken encoding
# only turns one word into <unk> ("Jo��o"), and line breaks count as spaces. Only
# train is cleaned: validation and test have to stay complete, otherwise the results
# would look better than they are.

## 9. Truncation

In [60]:
# Cut off texts above a maximum length? (Project Discussions, slide 10:
# "Truncation (yes/no and how much?)") Raw stories have at most 63 tokens
# (01_data_exploration.ipynb). Here the lengths as the models get them: the RNN reads
# O1 <sep> h <sep> O2 per candidate, architecture 1 averages each text on its own.
def n_ids(text: str) -> int:
    return len(encode(text, vocab))


rows = [row for split in splits.values() for row in split]
story_lengths = pd.Series(
    [
        n_ids(row["observation_1"]) + n_ids(row[h]) + n_ids(row["observation_2"]) + 2
        for row in rows
        for h in ["hypothesis_1", "hypothesis_2"]
    ]
)
text_lengths = pd.Series([n_ids(row[col]) for row in rows for col in TEXT_COLUMNS])
pct = [0.5, 0.99]
pd.DataFrame(
    {
        "story O1 <sep> h <sep> O2 (RNN)": story_lengths.describe(percentiles=pct),
        "single text (architecture 1)": text_lengths.describe(percentiles=pct),
    }
).round(1)

,story O1 <sep> h <sep> O2 (RNN),single text (architecture 1)
count,38666.0,77332.0
mean,28.6,8.8
std,5.5,3.0
min,11.0,0.0
50%,28.0,8.0
99%,43.0,17.0
max,60.0,34.0


In [61]:
# No truncation.
# The longest story the RNN reads has 60 tokens, 99% have at most 43 (table
# above). A batch of 128 such stories is about 10 MB on the GPU, and a GRU handles
# sequences of this length. Truncation usually cuts the end, and the end of our story
# is O2, the observation the hypothesis has to explain. Same as Project Discussions
# slide 10 (truncation: no).

## 10. Separator for the RNN

In [62]:
# What marks where O1, h and O2 begin and end in the story the
# RNN reads? (Project Discussions, slide 20: "Special tokens (separators)")
#   "."        GloVe's full stop: pretrained, but "." also occurs inside some texts
#   own <sep>  always unique, but needs a vector: a zero vector is the same as <unk>,
#              a trained one means the embeddings are not frozen any more, a fixed
#              random vector is unique and stays frozen
def story(row: dict, h: str, sep: list[int]) -> list[int]:
    o1, o2 = encode(row["observation_1"], vocab), encode(row["observation_2"], vocab)
    return o1 + sep + encode(row[h], vocab) + sep + o2


dot = vocab["."]
stories = [story(row, h, []) for row in rows for h in ["hypothesis_1", "hypothesis_2"]]
with_dot = sum(dot in s for s in stories)
share = 100 * with_dot / len(stories)
print(f'{with_dot} of {len(stories)} stories ({share:.2f}%) have a "." inside')

354 of 38666 stories (0.92%) have a "." inside


In [63]:
# What the RNN reads now (first training example, candidate 1)
print(" ".join(id_to_word[i] for i in story(train[0], "hypothesis_1", [SEP_IDX])))

a baby panda was lost in the woods <sep> he was never lost <sep> he was reunited with his parents , who also lived at the zoo


In [64]:
# Own <sep> token with a fixed random vector (same spread as the GloVe
# vectors, seed 42), frozen like all other embeddings.
# It always marks where O1, h and O2 begin and end; with "." this would not
# hold in about 1% of the stories. A zero vector would look exactly like <unk>, and a
# trained vector would mean the embeddings are not frozen any more, which the task
# requires.